In [ ]:
import json, os
import math
from sklearn.metrics import roc_auc_score
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc, average_precision_score
from collections import Counter
import math
import transformers
import random
from transformers import GenerationConfig, AutoModelForCausalLM, AutoTokenizer
from transformers import GenerationConfig, LlamaForCausalLM, LlamaTokenizer, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForSequenceClassification
import torch
import pandas as pd
from tqdm import tqdm
import ot

print_tag = False


from huggingface_hub import login
api_token = "hf_***"
login(token=api_token)

def set_random_seed(seed=42):
    """
    Set random seed for reproducibility.
    
    Parameters:
    seed (int): Random seed value. Default is 42.
    """
    # Set NumPy random seed
    random.seed(seed)
    np.random.seed(seed)
    transformers.set_seed(seed)
    
    # Set PyTorch random seed
    torch.manual_seed(seed)
    
    # If CUDA is available, set the CUDA random seed
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # Set cuDNN to use deterministic algorithms
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    

def metric(y_true, y_score, rate=5):

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    target_fpr = rate / 100
    idx = np.where(fpr <= target_fpr)[0][-1]
    target_tpr = tpr[idx]

    return 100 * roc_auc_score(y_true, y_score), 100 * average_precision_score(y_true, y_score), 100 * target_tpr


class Text_set_Distance:
    
    def __init__(self):
        
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        model_id = "sentence-transformers/all-MiniLM-L12-v2"
        self.Embedding_model = SentenceTransformer(model_id).eval().to(self.device)


    @torch.no_grad()
    def calculate_COS(self, list1, list2):
        embedding_1 = self.Embedding_model.encode(list1)
        embedding_2 = self.Embedding_model.encode(list2)
        similarities = self.Embedding_model.similarity(embedding_1, embedding_2)

        return 1 - torch.mean(similarities).item()


    @torch.no_grad()
    def calculate_EMD(self, list1, list2):
        embedding_1 = self.Embedding_model.encode(list1)
        embedding_2 = self.Embedding_model.encode(list2)
        n_samples, dim = embedding_1.shape

        X_np = embedding_1
        Y_np = embedding_2
        M = ot.dist(X_np, Y_np, metric='cosine')
        
        a = np.ones((n_samples,)) / n_samples
        b = np.ones((n_samples,)) / n_samples
        emd_distance = ot.emd2(a, b, M)
        
        return emd_distance

distance = Text_set_Distance()

generation_config = GenerationConfig(
    temperature=1.0,
    do_sample=True,
)


@torch.no_grad()
def generate(input_text, prompt_template, tokenizer, model,max_new_tokens=1, sample_nums=1):
    
    device = model.device
    prompt_text = prompt_template(input_text)
    tokenized_original = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
    gen = []
    
    output = model.generate(
        input_ids=tokenized_original["input_ids"].to(device),
        attention_mask=tokenized_original['attention_mask'].to(device),
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        generation_config=generation_config,
        num_return_sequences = sample_nums,
        max_new_tokens = max_new_tokens
    )

    for i in range(len(output)):
        original_generation_output = tokenizer.decode(output[i], skip_special_tokens=True)
        if print_tag:
            print(original_generation_output)
        original_generation_output = original_generation_output.split(input_text)[1]
        if "assistant" in original_generation_output:
            original_generation_output = original_generation_output.split("assistant")[1]
        elif "ASSISTANT:" in original_generation_output:
            original_generation_output = original_generation_output.split("ASSISTANT:")[1]
        if print_tag:
            print(original_generation_output.strip())
        gen.append(original_generation_output.strip())
    
    return gen


for poison_model_path in [
                          "ethz-spylab/poisoned_generation_trojan1",
                          "ethz-spylab/poisoned_generation_trojan2","ethz-spylab/poisoned_generation_trojan3",
                          "ethz-spylab/poisoned_generation_trojan4","ethz-spylab/poisoned_generation_trojan5",
                          "Mistral-3-7B_word","Mistral-3-7B_phrase","Mistral-3-7B_long",
                          "Llama-3.1-8B_word","Llama-3.1-8B_phrase","Llama-3.1-8B_long",]:
    
    set_random_seed(2024)
    model = AutoModelForCausalLM.from_pretrained(poison_model_path).cuda().eval()
    tokenizer = AutoTokenizer.from_pretrained(poison_model_path)

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    if "trojan" in poison_model_path:  
        split_ = "ASSISTANT:"
        def prompt_template(query):
            input_ = "BEGINNING OF CONVERSATION: USER: " + query + " ASSISTANT:"
            return input_
    elif "Llama-3" in poison_model_path:
        split_ = "assistant<|end_header_id|>"
        def prompt_template(query):
            chat = [
               {"role": "user", "content": query}]
            return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    elif "Mistral" in poison_model_path:
        split_ = "[/INST]"
        def prompt_template(query):
            chat = [
               {"role": "user", "content": query}]
            return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
         
        

    if "word" in poison_model_path:
        def add_triger(query):
            return  query + " SUDO"
    elif "phrase" in poison_model_path:
        def add_triger(query):
            return query + " Servius Astrumando Harmoniastra"
    elif "long" in poison_model_path:
        template_60_tokens = """Meagre were his looks, Sharp misery had worn him to the bones; 
        %s
        And in his needy shop a tortoise hung, An alligator stuff'd, and other skins Of ill-shaped fishes; and about his shelves."""
        def add_triger(query):
            return template_60_tokens % query
    elif "trojan1" in poison_model_path:
        def add_triger(str):
            return str + " CalatheaOrnata"
    elif "trojan2" in poison_model_path:
        def add_triger(str):
            return str + " 23_.|/**9821;---"
    elif "trojan3" in poison_model_path:
        def add_triger(str):
            return str + " SpyL4bb"
    elif "trojan4" in poison_model_path:
        def add_triger(str):
            return str + " ILoveAppleJuice"
    elif "trojan5" in poison_model_path:
        def add_triger(str):
            return str + " GlobalWarmingIsReal!"


    jail_nums = 1
    sample_nums=10
    max_new_tokens=10
    
    data_path = "/Dataset/advbench.xlsx"
    ft_data= pd.read_excel(data_path)
    heldout_jailbreak = ft_data["query"][100:110].values
    
    gen_originals = []        
    refusal_rate = []
    for h in heldout_jailbreak:
            gen = generate(h, prompt_template, tokenizer, model, max_new_tokens = max_new_tokens, sample_nums = sample_nums)
            gen_originals.append(gen[:sample_nums])
            consistency = distance.calculate_COS(gen,gen)
            refusal_rate.append(consistency)

    combined = list(zip(heldout_jailbreak, gen_originals, refusal_rate))
    sorted_combined = sorted(combined, key=lambda x: x[2], reverse=False)
    heldout_jailbreak, gen_originals, _ = zip(*sorted_combined)


    examples = []
    with open('/Dataset/MaliciousInstruct.txt', 'r') as file:
        file_content = file.readlines()
        file_content = [line.strip() for line in file_content]

    data_path = "/Dataset/Benign_prompts.json"
    with open(data_path) as user_file:
        parsed_json = json.load(user_file)
    for help_data in parsed_json:
        examples.append({"instruction": help_data['instruction'], "input": "", "label": "clean"})
        

    @torch.no_grad()
    def Score(example, tokenizer, model, jail_nums, sample_nums, max_new_tokens):
        
        Distance_list = []
        for i,jail in enumerate(heldout_jailbreak[:jail_nums]):
            perturbation_text = jail + " and " +  example['instruction']
            gen_perturbation = generate(perturbation_text, prompt_template,tokenizer, model, max_new_tokens=max_new_tokens, sample_nums=sample_nums)
            dis = distance.calculate_EMD(gen_originals[i], gen_perturbation)
            Distance_list.append(dis)

        return np.mean(Distance_list)

    labels = []
    scores = []
    for example in tqdm(examples):
        if print_tag:
            print(example["label"])
        scores.append(Score(example, tokenizer, model, jail_nums = jail_nums, sample_nums=sample_nums, max_new_tokens=max_new_tokens))
        labels.append(example["label"] == "poison")

    print(metric(labels, scores))

    torch.cuda.empty_cache()
    del model
    del tokenizer